In [1]:

# ================================================================
# PROJETO DE ZOONOSES
# CONSTRUÇÃO DO BANCO DE DADOS SQLITE
# VERSÃO 2.0 — SEM GOOGLE DRIVE
#
# - Cria a estrutura do banco.
# - Preserva dados existentes.
# - Mantém controle de sequências.
# - Prepara as tabelas para o tratamento dos dados brutos.
# ================================================================


# 1. IMPORTAÇÃO DAS BIBLIOTECAS

import sqlite3
import os
from datetime import datetime


# 2. CONFIGURAÇÃO DO PROJETO

PASTA_PROJETO = "/content/PROJETO_ZOONOSES"

os.makedirs(PASTA_PROJETO, exist_ok=True)

CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)


# 3. CONEXÃO COM SQLITE

conexao = sqlite3.connect(
    CAMINHO_BANCO,
    timeout=60
)

cursor = conexao.cursor()

cursor.execute("PRAGMA foreign_keys = ON")
cursor.execute("PRAGMA busy_timeout = 60000")


# 4. CRIAÇÃO DAS TABELAS

cursor.executescript("""

-- ----------------------------------------------------------
-- TABELA 1: LOTES DE DADOS
-- Registra cada geração ou importação de dados.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS lotes (

    id_lote INTEGER PRIMARY KEY AUTOINCREMENT,

    tipo_lote TEXT NOT NULL,

    quantidade_informada INTEGER,

    quantidade_gerada INTEGER,

    data_criacao TEXT NOT NULL,

    descricao TEXT,

    status TEXT DEFAULT 'Registrado'

);


-- ----------------------------------------------------------
-- TABELA 2: REGISTROS BRUTOS
-- Preserva os dados originais, inclusive os problemáticos.
-- Os dados são armazenados em formato JSON.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS registros_brutos (

    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,

    id_lote INTEGER,

    conjunto TEXT NOT NULL,

    dados_json TEXT NOT NULL,

    data_recebimento TEXT NOT NULL,

    FOREIGN KEY (id_lote)
        REFERENCES lotes(id_lote)

);


-- ----------------------------------------------------------
-- TABELA 3: ANIMAIS
-- Armazena os animais após validação e tratamento.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS animais (

    id_animal TEXT PRIMARY KEY,

    nome TEXT,

    especie TEXT,

    raca TEXT,

    sexo TEXT,

    municipio TEXT,

    peso_kg REAL,

    status TEXT,

    vacinas_faltantes TEXT,

    situacao_vacinacao TEXT,

    imagem TEXT,

    revisao_peso TEXT,

    data_cadastro TEXT,

    data_acolhimento TEXT,

    motivo_acolhimento TEXT,

    origem_acolhimento TEXT,

    id_lote INTEGER,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_lote)
        REFERENCES lotes(id_lote)

);


-- ----------------------------------------------------------
-- TABELA 4: VACINAÇÕES
-- Um animal pode possuir várias vacinações.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS vacinacoes (

    id_vacinacao INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT NOT NULL,

    nome_vacina TEXT NOT NULL,

    data_aplicacao TEXT,

    proxima_dose TEXT,

    situacao TEXT,

    observacoes TEXT,

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 5: DIAGNÓSTICOS
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS diagnosticos (

    id_diagnostico INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT NOT NULL,

    data_diagnostico TEXT,

    diagnostico TEXT,

    descricao TEXT,

    situacao TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 6: MAUS-TRATOS
-- Permite registrar ocorrências sem animal identificado.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS maus_tratos (

    id_ocorrencia INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT,

    municipio TEXT,

    data_ocorrencia TEXT,

    tipo_ocorrencia TEXT,

    descricao TEXT,

    situacao TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 7: ÓBITOS E EUTANÁSIA
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS obitos_eutanasias (

    id_evento INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT,

    tipo_evento TEXT NOT NULL,

    data_evento TEXT,

    municipio TEXT,

    motivo TEXT,

    observacoes TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 8: AUDITORIA
-- Registra inclusões, atualizações e correções.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS auditoria (

    id_auditoria INTEGER PRIMARY KEY AUTOINCREMENT,

    entidade TEXT NOT NULL,

    id_registro TEXT,

    operacao TEXT NOT NULL,

    descricao TEXT,

    data_operacao TEXT NOT NULL

);


-- ----------------------------------------------------------
-- TABELA 9: CONTROLE DE SEQUÊNCIAS
-- Mantém a numeração dos animais entre os lotes.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS controle_sequencias (

    nome TEXT PRIMARY KEY,

    proximo_valor INTEGER NOT NULL

);


-- ----------------------------------------------------------
-- ÍNDICES PARA MELHORAR AS CONSULTAS
-- ----------------------------------------------------------

CREATE INDEX IF NOT EXISTS idx_animais_municipio
ON animais(municipio);

CREATE INDEX IF NOT EXISTS idx_animais_especie
ON animais(especie);

CREATE INDEX IF NOT EXISTS idx_diagnosticos_animal
ON diagnosticos(id_animal);

CREATE INDEX IF NOT EXISTS idx_maus_tratos_animal
ON maus_tratos(id_animal);

CREATE INDEX IF NOT EXISTS idx_brutos_lote
ON registros_brutos(id_lote);

CREATE INDEX IF NOT EXISTS idx_brutos_conjunto
ON registros_brutos(conjunto);


-- ----------------------------------------------------------
-- INICIALIZAÇÃO DO CONTROLE DE SEQUÊNCIAS
-- ----------------------------------------------------------

INSERT OR IGNORE INTO controle_sequencias
(nome, proximo_valor)
VALUES ('animais', 1);

""")


# 5. IDENTIFICAÇÃO DA VERSÃO DO BANCO

cursor.execute("PRAGMA user_version = 2")


# 6. SALVAR AS ALTERAÇÕES

conexao.commit()


# 7. VERIFICAR AS TABELAS CRIADAS

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

tabelas = cursor.fetchall()

print("\n" + "=" * 60)
print("BANCO DE DADOS DE ZOONOSES")
print("=" * 60)

print("\nTABELAS DISPONÍVEIS:")

for tabela in tabelas:
    print(" -", tabela[0])


# 8. VERIFICAR QUANTIDADE DE ANIMAIS

cursor.execute("""
SELECT COUNT(*)
FROM animais
""")

total_animais = cursor.fetchone()[0]

print("\nTOTAL DE ANIMAIS CADASTRADOS:", total_animais)


# 9. VERIFICAR CAMINHO DO BANCO

print("\nLOCAL DO BANCO:")
print(CAMINHO_BANCO)


# 10. FINALIZAÇÃO

print("\nESTRUTURA DO BANCO CRIADA E VERIFICADA!")

conexao.close()



BANCO DE DADOS DE ZOONOSES

TABELAS CRIADAS:
 - animais
 - auditoria
 - diagnosticos
 - lotes
 - maus_tratos
 - obitos_eutanasias
 - registros_brutos
 - sqlite_sequence
 - vacinacoes

TOTAL DE ANIMAIS CADASTRADOS: 0

LOCAL DO BANCO:
/content/PROJETO_ZOONOSES/banco_zoonoses_v2.db

ESTRUTURA CRIADA E SALVA COM SUCESSO!


In [2]:

# ================================================================
# PROJETO DE ZOONOSES
# ATUALIZAÇÃO E VERIFICAÇÃO DO BANCO
# ================================================================

import sqlite3

conexao = sqlite3.connect(
    CAMINHO_BANCO,
    timeout=60
)

cursor = conexao.cursor()

cursor.execute("PRAGMA foreign_keys = ON")
cursor.execute("PRAGMA busy_timeout = 60000")

try:

    # 1. VERIFICAR COLUNAS DA TABELA ANIMAIS

    cursor.execute("PRAGMA table_info(animais)")

    colunas = {
        coluna[1]
        for coluna in cursor.fetchall()
    }

    novas_colunas = {
        "data_acolhimento": "TEXT",
        "motivo_acolhimento": "TEXT",
        "origem_acolhimento": "TEXT"
    }

    for nome, tipo in novas_colunas.items():

        if nome not in colunas:

            cursor.execute(
                f"ALTER TABLE animais ADD COLUMN {nome} {tipo}"
            )

            print(f"Coluna adicionada: {nome}")

        else:

            print(f"Coluna já existente: {nome}")


    # 2. GARANTIR CONTROLE DE SEQUÊNCIAS

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS controle_sequencias (

            nome TEXT PRIMARY KEY,

            proximo_valor INTEGER NOT NULL

        )
    """)

    cursor.execute("""
        INSERT OR IGNORE INTO controle_sequencias
        (nome, proximo_valor)
        VALUES ('animais', 1)
    """)


    # 3. VERIFICAR A SEQUÊNCIA ATUAL

    cursor.execute("""
        SELECT nome, proximo_valor
        FROM controle_sequencias
    """)

    sequencias = cursor.fetchall()

    print("\nCONTROLE DE SEQUÊNCIAS:")

    for nome, valor in sequencias:
        print(f"{nome}: próximo valor = {valor}")


    # 4. VERIFICAR CHAVES ESTRANGEIRAS

    cursor.execute("PRAGMA foreign_key_check")

    problemas_relacionamento = cursor.fetchall()

    if problemas_relacionamento:

        print("\nATENÇÃO: foram encontrados problemas de relacionamento.")

        for problema in problemas_relacionamento:
            print(problema)

    else:

        print("\nIntegridade referencial verificada.")


    # 5. SALVAR

    conexao.commit()

    print("\nATUALIZAÇÃO E VERIFICAÇÃO CONCLUÍDAS!")

except Exception:

    conexao.rollback()
    raise

finally:

    conexao.close()


Coluna adicionada: data_acolhimento
Coluna adicionada: motivo_acolhimento
Coluna adicionada: origem_acolhimento

Atualização concluída!
